### def hàm chính

In [ ]:

import numpy as np
import pandas as pd
import QuantLib as ql

In [ ]:

def calculate_continuous_zyc_advanced(df_input, daycount='Actual/365', convention='ModifiedFollowing'):
    """
    Tính toán chuỗi thời gian Zero-Coupon liên tục trực tiếp từ YTM Annual sử dụng QuantLib.
    
    Tham số:
    - df_input: DataFrame chuỗi thời gian (Index là Date, các cột dạng '3m', '6m', '1y'...)
    - daycount: Quy ước đếm ngày ('Actual/365', 'Actual/360', '30/360', 'Actual/Actual')
    - convention: Quy ước điều chỉnh ngày lễ ('Following', 'ModifiedFollowing', 'Preceding')
    """

    calendar = ql.Vietnam() #đổi theo calendar trong package
    
    # Cấu hình Quy ước đếm ngày
    day_count_map = {
        'actual/365': ql.Actual365Fixed(),
        'act/365': ql.Actual365Fixed(),
        'actual/360': ql.Actual360(),
        'act/360': ql.Actual360(),
        '30/360': ql.Thirty360(ql.Thirty360.BondBasis),
        'actual/actual': ql.ActualActual(ql.ActualActual.ISDA)
    }
    day_counter = day_count_map.get(daycount.lower(), ql.Actual365Fixed())
    
    # Cấu hình Quy ước điều chỉnh ngày lễ 
    convention_map = {
        'following': ql.Following,
        'modifiedfollowing': ql.ModifiedFollowing,
        'preceding': ql.Preceding
    }
    biz_convention = convention_map.get(convention.lower(), ql.ModifiedFollowing)
    
    # 2. Phân tích các cột kỳ hạn (Tenors) thành đối tượng ql.Period
    original_columns = df_input.columns
    ql_periods = []
    
    for col in original_columns:
        clean_name = str(col).strip().lower()
        value = int(''.join(filter(str.isdigit, clean_name)))
        
        if 'm' in clean_name:
            ql_periods.append(ql.Period(value, ql.Months))
        elif 'y' in clean_name:
            ql_periods.append(ql.Period(value, ql.Years))
        else:
            raise ValueError(f"Không nhận diện được định dạng kì hạn của cột: {col}")
            
    # Khởi tạo bảng kết quả đầu ra (giữ đúng kích thước)
    df_output = pd.DataFrame(index=df_input.index, columns=original_columns)
    
    # 3. Duyệt qua từng dòng chuỗi thời gian (từng ngày cụ thể)
    for date_idx, row in df_input.iterrows():
        # Chuyển đổi Ngày đánh giá (Evaluation Date) từ Pandas sang QuantLib
        eval_date = ql.Date(date_idx.day, date_idx.month, date_idx.year)
        
        # Tính toán mốc ngày đáo hạn thực tế cho từng kỳ hạn sau khi qua bộ lọc Holiday và Quy ước
        # Dùng lịch Vietnam() để tịnh tiến ngày (advance) dựa theo quy ước (biz_convention)
        maturities = [calendar.advance(eval_date, period, biz_convention) for period in ql_periods]
        
        # Tính year_fraction thực tế giữa Ngày đáo hạn chuẩn và Ngày đánh giá gốc
        tenors = np.array([day_counter.yearFraction(eval_date, mat) for mat in maturities])
        ytms = row.values
        
        # Sắp xếp cục bộ để thuật toán Bootstrap chạy tuyến tính chính xác từ ngắn đến dài
        sort_idx = np.argsort(tenors)
        sorted_tenors = tenors[sort_idx]
        sorted_ytms = ytms[sort_idx]
        
        sorted_zero_rates = np.zeros(len(sorted_tenors))
        
        # 4. Thực thi lõi Bootstrap dựa trên ma trận year fraction 
        for i, t in enumerate(sorted_tenors):
            y = sorted_ytms[i]
            
            # Kỳ hạn nhỏ hơn hoặc bằng 1 năm (Dòng tiền đơn)
            if t <= 1.0:
                sorted_zero_rates[i] = np.log(1 + y * t) / t
            
            # Kỳ hạn dài (Bootstrap rút gốc từng kỳ Annual lùi dần từ điểm t)
            else:
                sum_pv_coupons = 0.0
                # Lịch trả coupon lùi dần 1 năm một lần từ ngày đáo hạn thực tế về ngày hiện tại
                coupon_times = np.arange(t - 1.0, 0, -1.0)[::-1]
                
                if len(coupon_times) > 0 and i > 0:
                    for ct in coupon_times:
                        # Nội suy tuyến tính lãi suất liên tục tại điểm ct từ các điểm ngắn hơn đã tính trước đó
                        r_ct = np.interp(ct, sorted_tenors[:i], sorted_zero_rates[:i])
                        sum_pv_coupons += y * np.exp(-r_ct * ct)
                        
                # Phương trình Par-Bond quy ước lãi suất coupon hàng năm nhận gốc cuối kì
                target_pv = (1.0 - sum_pv_coupons) / (1.0 + y)
                
                if target_pv <= 0:
                    sorted_zero_rates[i] = sorted_zero_rates[i-1] if i > 0 else 0.0
                else:
                    sorted_zero_rates[i] = -np.log(target_pv) / t
                    
        # Trả lại thứ tự cột gốc ban đầu dựa trên index xáo trộn ban đầu
        unsort_idx = np.argsort(sort_idx)
        df_output.loc[date_idx] = sorted_zero_rates[unsort_idx]
        
    return df_output.astype(float)




## 2. Test


In [ ]:

# Tạo chuỗi thời gian mẫu với index là Datetime của Pandas
date_series = pd.to_datetime(["2026-04-30", "2026-05-01", "2026-05-04"]) # Gồm cả ngày lễ 1/5 và cuối tuần

data_input = {
    "3m": [0.021, 0.022, 0.020],
    "6m": [0.023, 0.024, 0.022],
    "1y": [0.025, 0.026, 0.024],
    "2y": [0.028, 0.029, 0.027],
    "3y": [0.032, 0.033, 0.031],
    "5y": [0.037, 0.039, 0.036],
    "10y": [0.045, 0.046, 0.044]
}

df_ytm = pd.DataFrame(data_input, index=date_series)
print("DỮ LIỆU INPUT CHUỖI THỜI GIAN YTM GỐC:")
print(df_ytm)

# Chạy hàm với bộ cấu hình như sau
df_zyc_final = calculate_continuous_zyc_advanced(
    df_input=df_ytm, 
    daycount='Actual/365', 
    convention='ModifiedFollowing'
)

print("\nDỮ LIỆU KẾT QUẢ ZERO-COUPON LIÊN TỤC (GIỮ NGUYÊN KÍCH THƯỚC):")
print(df_zyc_final)